# TSLA 36-run sweep — analysisEvery number in `analyses/tsla_200days_report.html`, recomputed here from source so eachone can be checked and changed.**Data.** `results.csv` (36 configurations) and `additions.csv` (360 proposed ontologyadditions), both produced by `pull_and_analyze.py`, which pulls the runs from the MLflowtracking server. Re-run that script to refresh; this notebook only reads the CSVs.**Scope.** The 36 runs are the TSLA half of the balanced design, on the 200-day window(2022-05-02 → 2022-11-18), all after the `36f4f88` path-embedding fix. Runs before that fixhad a silently empty path feature block and must not be pooled with these.

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom scipy import statsruns = pd.read_csv("results.csv")adds = pd.read_csv("additions.csv")FACTORS = ["lookback_days", "steps", "chain_hops", "evolution_prompt"]print(f"{len(runs)} configurations, {len(adds)} addition records")runs.head(3)

## A · What was runFour hyperparameters varied at once. The full grid is 4 x 3 x 3 x 3 = 108 combinations;36 were run — one third — chosen so that comparisons stay fair.

In [ ]:
# Everything that was held fixed should have exactly one distinct value.fixed = runs.nunique()print("varying:", list(fixed[fixed > 1].index[fixed[fixed > 1].index.isin(FACTORS)]))print()for f in FACTORS:    print(f"{f:18} {sorted(runs[f].unique(), key=str)}   n per level = {runs[f].value_counts().unique()}")

In [ ]:
# Pairwise balance: for every PAIR of factors, does every level combination appear,# and equally often? Counting observed combinations alone would hide a missing cell,# so build the full grid and check for zeros explicitly.from itertools import combinationsfor a, b in combinations(FACTORS, 2):    grid = pd.crosstab(runs[a], runs[b])    counts = sorted(set(grid.values.ravel()))    print(f"{a:17} x {b:18} cells={grid.size:>2}  missing={int((grid == 0).sum().sum())}  counts={counts}")

Every pair is complete and each combination appears an equal number of times (3 or 4,depending on how many levels the two factors have). That is what lets us read a factor'seffect straight off its group means: comparing `steps=3` against `steps=7`, both groupshold an identical mix of every other setting, so no adjustment is needed.

## B · ResultsEach run ends at its **final accepted ontology**. `baseline_auc` is the article-textbaseline computed on the same days.

In [ ]:
cols = ["final_auc", "baseline_auc", "delta"] + FACTORS + ["path_mean_norm"]runs.sort_values("final_auc", ascending=False)[cols].head(10).reset_index(drop=True)

In [ ]:
runs[["final_auc", "baseline_auc", "delta", "brier_score"]].describe().T[    ["mean", "std", "50%", "min", "max"]].round(4)

In [ ]:
beat = (runs.delta > 0).sum()lost = (runs.delta < 0).sum()print(f"beat the text baseline: {beat}   fell below it: {lost}   tied: {(runs.delta == 0).sum()}")print(f"mean difference {runs.delta.mean():+.4f}   range {runs.delta.min():+.4f} to {runs.delta.max():+.4f}")

### The two arms share no features`build_day_feature_vector` skips article text entirely ("using structural signals only"),so the evolved arm is path (384) + subgraph (96) + topology (8) = **488 structuraldimensions with no text**, while the baseline is mean-pooled article embeddings with nograph. `delta` is therefore a contrast between two disjoint representations, not anablation of the ontology. No configuration in this sweep tested *text + graph* against*text*, so nothing here shows the knowledge graph adding to what the article text alreadycarries.One consequence worth checking directly: because the baseline pools articles over the samelookback window as the graph arm, `lookback_days` moves **both** arms.

In [ ]:
by_lb = runs.groupby("lookback_days").agg(    n=("final_auc", "size"),    distinct_baselines=("baseline_auc", "nunique"),    baseline=("baseline_auc", "first"),    final_auc=("final_auc", "mean"),    delta=("delta", "mean"),    beat=("delta", lambda s: (s > 0).sum()),).sort_values("delta", ascending=False)by_lb.round(4)

One distinct baseline per lookback level, identical across all 9 runs at that level — sothe baseline is fully deterministic, and the variation is the lookback factor rather thannoise. It also means lookback is the only factor where ranking by absolute AUC and rankingby improvement-over-text disagree: `lookback=20` is second by final AUC but last by delta,and loses 6 of its 9 head-to-heads.

## C · Do path-chain features help?`path_mean_norm` is the average Euclidean length of each day's 384-dimension path vector:0 means the block is entirely zeros, ~0.79 means most days carry real chain embeddings.Treating it as a continuous measure of realised path signal avoids the confound in ahop-level comparison, where deep-hop runs are path-sparse *because* long chains areunreachable at this graph density.

In [ ]:
rho, p = stats.spearmanr(runs.path_mean_norm, runs.final_auc)slope = np.polyfit(runs.path_mean_norm, runs.final_auc, 1)[0]print(f"Spearman rho = {rho:+.3f} (p = {p:.2f})    OLS slope = {slope:+.3f}")fig, ax = plt.subplots(figsize=(8, 5))for s, grp in runs.groupby("steps"):    ax.scatter(grp.path_mean_norm, grp.final_auc, s=70, alpha=0.8, label=f"steps = {s}")xs = np.linspace(0, runs.path_mean_norm.max(), 50)ax.plot(xs, np.polyval(np.polyfit(runs.path_mean_norm, runs.final_auc, 1), xs),        color="grey", ls="--", label="fit")ax.set_xlabel("path_mean_norm  (realised path signal)")ax.set_ylabel("final AUC")ax.set_title(f"Dose-response: more path signal does not mean better AUC (rho = {rho:+.3f})")ax.legend()plt.show()

In [ ]:
# How often is the path block empty at all, and does hop depth determine it?runs.assign(empty=runs.path_mean_norm == 0).groupby("chain_hops").agg(    n=("final_auc", "size"),    empty_path_blocks=("empty", "sum"),    mean_path_signal=("path_mean_norm", "mean"),    mean_auc=("final_auc", "mean"),).round(4)

Hop depth shifts the *probability* of an empty block rather than determining it. Pathsignal falls by an order of magnitude across hop levels while AUC does not follow.

## D · Differential analysis by factor

In [ ]:
grand, sd = runs.final_auc.mean(), runs.final_auc.std()print(f"grand mean {grand:.4f}   between-run SD {sd:.4f}")tbl = []for f in FACTORS:    g = runs.groupby(f).final_auc.agg(["size", "mean", "std"])    for level, row in g.sort_values("mean", ascending=False).iterrows():        tbl.append({"factor": f, "level": level, "n": int(row["size"]),                    "mean_auc": row["mean"], "sd": row["std"],                    "dev_from_grand": row["mean"] - grand,                    "in_SD_units": abs(row["mean"] - grand) / sd})tbl = pd.DataFrame(tbl)tbl.round(4)

In [ ]:
print(f"largest separation from the grand mean: {tbl.in_SD_units.max():.2f} SD "      f"({tbl.loc[tbl.in_SD_units.idxmax(), 'factor']} = "      f"{tbl.loc[tbl.in_SD_units.idxmax(), 'level']})")fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)for ax, f in zip(axes.flat, FACTORS):    g = runs.groupby(f).final_auc.agg(["mean", "std"])    x = range(len(g))    ax.axhspan(grand - sd, grand + sd, alpha=0.12, label="grand mean ± 1 SD")    ax.axhline(grand, ls=":", color="black")    ax.errorbar(x, g["mean"], yerr=g["std"], fmt="o", capsize=5, markersize=9)    ax.set_xticks(list(x))    ax.set_xticklabels(g.index, rotation=15)    ax.set_title(f)    ax.set_ylabel("mean AUC ± SD")axes.flat[0].legend(loc="lower right", fontsize=9)fig.suptitle("Every factor level sits inside the band the runs already occupy", y=1.0)fig.tight_layout()plt.show()

In [ ]:
# How much of the run-to-run variation do the factors actually explain? The design is# balanced, so effects are additive and the fitted value is just the sum of level effects.effects = {f: runs.groupby(f).final_auc.mean() - grand for f in FACTORS}fitted = grand + sum(runs[f].map(effects[f]) for f in FACTORS)resid = runs.final_auc - fittedprint(f"variance of final AUC   {runs.final_auc.var():.5f}")print(f"residual variance       {resid.var():.5f}")print(f"explained by 4 factors  {100 * (1 - resid.var() / runs.final_auc.var()):.1f}%")print()for f in FACTORS:    ss = (runs.groupby(f).final_auc.agg(["size", "mean"])          .pipe(lambda g: (g["size"] * (g["mean"] - grand) ** 2).sum()))    print(f"  {f:18} {100 * ss / (runs.final_auc.var() * (len(runs) - 1)):.1f}%")

Roughly three quarters of why one run beats another has nothing to do with anyhyperparameter. Note `steps` is the weakest of the four, which is the sweep's core claimabout evolution depth.

## E · Ontology-change analysisEach candidate proposes **one node type and one relationship together**, so the pair is theunit of intervention — not the node.

In [ ]:
print(f"node types: {adds.node.nunique()}   distinct (node, relationship) pairs: "      f"{adds.groupby(['node', 'relationship']).ngroups}")multi = adds.groupby("node").relationship.nunique()print(f"node types proposed with more than one relationship: {(multi > 1).sum()}")adds[adds.node == "RegulatoryBody"].relationship.value_counts()

### Why not score against the run's current bestThe obvious measure is a candidate's AUC minus the run's current best. It is unusable: theacceptance gate makes that reference the *maximum seen so far*, so almost any candidatescores negative whatever its quality — regression to the maximum, not evidence.

In [ ]:
adds["is_step1"] = adds.step == 1adds.groupby(adds.is_step1.map({True: "step 1 (ref = text baseline)",                                False: "steps 2+ (ref = best so far)"})).agg(    n=("delta_auc", "size"),    mean_delta=("delta_auc", "mean"),    pct_negative=("delta_auc", lambda s: 100 * (s < 0).mean()),).round(3)

Within a single step, though, both candidates are built on the **same** graph and scored onthe same days. Comparing them to each other carries no reference bias at all.

In [ ]:
# Each candidate against the mean of its own step.adds["within_step"] = adds.auc - adds.groupby(["run_id", "step"]).auc.transform("mean")pairs = (adds.groupby(["node", "relationship"])         .within_step.agg(n="size", effect="mean", sd="std")         .query("n >= 8").sort_values("effect", ascending=False))pairs.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))labels = [f"{n} —{r}→" for n, r in pairs.index]ax.barh(labels, pairs.effect, color=["tab:green" if v > 0 else "tab:red" for v in pairs.effect])ax.axvline(0, color="black", lw=1)ax.invert_yaxis()ax.set_xlabel("AUC relative to the alternative proposed at the same step")ax.set_title(f"Effect per addition — {len(pairs)} pairs with n ≥ 8 "             f"({int(pairs.n.sum())} of {len(adds)} records)")fig.tight_layout()plt.show()

The measure is zero-sum by construction: it ranks additions against each other and saysnothing about whether additions help in absolute terms.

In [ ]:
adds.groupby("step").agg(    proposed=("status", "size"),    accepted=("status", lambda s: (s == "accepted").sum()),    accept_rate=("status", lambda s: 100 * (s == "accepted").mean()),).round(1)

Half of first-step candidates are accepted, then 6–15% from step 2 onward — the ontologysaturates early. Step 1's rate is not a quality signal: `_should_accept_step` returns `True`whenever there is no previously accepted step, so step 1 is accepted unconditionally inevery run.## What is not measured- **An addition's absolute effect**, which would be AUC(base + addition) − AUC(base). The  base structural graph is never scored — step 0 evaluates only the article-text baseline.- **Whether the graph adds to text**, which needs a text + graph arm (see section B).- **Training AUC**, so the train/validation gap is unknown. With 488 features against 98  training days, that gap is the diagnostic this design most needs.- **Replication variance** — all 36 configurations are distinct, so no two runs share  settings and run-to-run noise was never observed directly.